# NB-02 · Descriptive Statistics & Health Metrics
## Module 01 — Health Analytics with Python
---
**Learning objectives**
- Compute prevalence and incidence rates with proper denominators
- Produce stratified summaries by age group, sex, and payer
- Interpret skewed distributions for LOS and hospital charges
- Apply age-standardisation (direct method) to compare populations
- Build a publication-ready descriptive "Table 1"

**Estimated time:** 1.5 hours  
**Level:** Beginner → Intermediate  
**Libraries:** `pandas`, `numpy`, `scipy.stats`, `tableone`


## 1. Setup & Data Reconstruction

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Rebuild the same synthetic dataset from NB-01 (identical seed)
np.random.seed(42)
N = 500

ages      = np.random.normal(62, 16, N).clip(18, 95).astype(int)
sexes     = np.random.choice(['M','F'], N, p=[0.48,0.52])
payers    = np.random.choice(
    ['Medicare','Medicaid','Private','Self-pay','Other'],
    N, p=[0.40,0.22,0.28,0.07,0.03])
drg_codes = np.random.choice([470,291,392,690,871,193,247,603], N)
drg_labels = {470:'Major joint replacement',291:'Heart failure & shock',
              392:'Esophagitis/misc GI',690:'Kidney/UTI',871:'Septicemia',
              193:'Simple pneumonia',247:'Perc cardiovasc w stent',603:'Cellulitis'}
los_days  = np.random.gamma(2.5, 1.8, N).clip(1, 30).astype(int)
charges   = (los_days * np.random.uniform(1800, 4200, N)).round(2)
charges[np.random.rand(N) < 0.08] = np.nan
primary_dx = np.random.choice(
    ['E11.9','I10','N18.3','I50.9','J18.9','A41.9','M16.11','N39.0'], N)
admit_type = np.random.choice(
    ['Emergency','Elective','Urgent','Newborn'], N, p=[0.52,0.30,0.16,0.02])

# Add comorbidities
np.random.seed(99)
df = pd.DataFrame({
    'patient_id'   : [f'PT-{str(i).zfill(5)}' for i in range(1, N+1)],
    'age'          : ages,
    'sex'          : sexes,
    'payer'        : payers,
    'drg_code'     : drg_codes,
    'drg_label'    : [drg_labels[d] for d in drg_codes],
    'primary_dx'   : primary_dx,
    'admit_type'   : admit_type,
    'los_days'     : los_days,
    'total_charge' : charges,
    'diabetes'     : np.random.binomial(1, 0.32, N),
    'hypertension' : np.random.binomial(1, 0.45, N),
    'ckd'          : np.random.binomial(1, 0.15, N),
    'heart_failure': np.random.binomial(1, 0.18, N),
    'readmitted_30': np.random.binomial(1, 0.14, N),
})

# Age groups (standard epidemiological groupings)
bins   = [17, 44, 64, 74, 95]
labels = ['18-44','45-64','65-74','75+']
df['age_group'] = pd.cut(df['age'], bins=bins, labels=labels)

print(f"Dataset ready: {df.shape[0]} discharges, {df.shape[1]} variables")
df.head(4)


## 2. Prevalence & Incidence — Core Epidemiological Measures

| Measure | Formula | When to use |
|---------|---------|-------------|
| **Prevalence** | Cases / Total population | Existing conditions at a point in time |
| **Incidence proportion** | New cases / At-risk population | New events over a defined period |
| **Rate per N** | (Cases / Population) × N | Standardised comparisons across cohorts |


In [ ]:
# --- 2.1  Crude prevalence of comorbidities ---
comorbidities = ['diabetes','hypertension','ckd','heart_failure']
prev_table = pd.DataFrame({
    'n_cases'    : df[comorbidities].sum(),
    'prevalence' : (df[comorbidities].mean() * 100).round(1),
})
prev_table.index.name = 'condition'
prev_table.columns = ['N cases', 'Prevalence (%)']
print("Crude prevalence in discharge cohort (N=500):")
display(prev_table)


In [ ]:
# --- 2.2  Readmission rate per 100 discharges ---
total_discharges = len(df)
readmissions     = df['readmitted_30'].sum()
rate_per_100     = readmissions / total_discharges * 100

print(f"30-day readmissions : {readmissions}")
print(f"Total discharges    : {total_discharges}")
print(f"Readmission rate    : {rate_per_100:.1f} per 100 discharges")


In [ ]:
# --- 2.3  95% confidence interval for a proportion (Wilson method) ---
from scipy.stats import norm

def wilson_ci(k, n, alpha=0.05):
    """Wilson score interval — preferred for proportions near 0 or 1."""
    z = norm.ppf(1 - alpha/2)
    p_hat = k / n
    denom = 1 + z**2 / n
    centre = (p_hat + z**2 / (2*n)) / denom
    half_width = (z * np.sqrt(p_hat*(1-p_hat)/n + z**2/(4*n**2))) / denom
    return round((centre - half_width)*100, 1), round((centre + half_width)*100, 1)

lo, hi = wilson_ci(readmissions, total_discharges)
print(f"Readmission rate: {rate_per_100:.1f}% (95% CI: {lo}% – {hi}%)")


## 3. Stratified Summaries

In [ ]:
# --- 3.1  Readmission rate by payer ---
strat_payer = (
    df.groupby('payer')
      .agg(
          n_discharges = ('patient_id','count'),
          n_readmit    = ('readmitted_30','sum'),
          mean_los     = ('los_days','mean'),
          mean_charge  = ('total_charge','mean'),
      )
      .assign(readmit_rate=lambda x: (x.n_readmit / x.n_discharges * 100).round(1),
              mean_los=lambda x: x.mean_los.round(1),
              mean_charge=lambda x: x.mean_charge.round(0))
      .sort_values('readmit_rate', ascending=False)
)
strat_payer.columns = ['N','Readmissions','Mean LOS (days)','Mean Charge ($)','Readmit rate (%)']
print("Stratified by payer:")
display(strat_payer)


In [ ]:
# --- 3.2  Stratified by age group and sex (2×4 pivot) ---
pivot = (
    df.pivot_table(
        values='readmitted_30',
        index='age_group',
        columns='sex',
        aggfunc=['mean','count']
    )
)
pivot.columns = ['N (F)','N (M)','Rate (F)','Rate (M)']
pivot[['Rate (F)','Rate (M)']] = (pivot[['Rate (F)','Rate (M)']] * 100).round(1)
print("Readmission rate (%) by age group and sex:")
display(pivot)


In [ ]:
# --- 3.3  LOS summary by admission type ---
los_summary = (
    df.groupby('admit_type')['los_days']
      .describe(percentiles=[.25,.50,.75,.90])
      .round(1)
)
print("LOS distribution by admission type:")
display(los_summary)


## 4. Interpreting Skewed Clinical Distributions

In [ ]:
# --- 4.1  LOS skewness and kurtosis ---
from scipy.stats import skew, kurtosis

los = df['los_days'].dropna()
chg = df['total_charge'].dropna()

for name, series in [('LOS (days)', los), ('Total charge ($)', chg)]:
    print(f"{name}")
    print(f"  Mean   : {series.mean():.2f}")
    print(f"  Median : {series.median():.2f}")
    print(f"  Std    : {series.std():.2f}")
    print(f"  Skew   : {skew(series):.3f}  {'(right-skewed — median is preferred)' if skew(series) > 0.5 else ''}")
    print(f"  Kurtosis: {kurtosis(series):.3f}")
    print()


In [ ]:
# --- 4.2  Log-transformation for skewed charge data ---
df['log_charge'] = np.log1p(df['total_charge'])

print("Before log-transform — charge skewness:", round(skew(chg), 3))
print("After  log-transform — skewness        :", round(skew(df['log_charge'].dropna()), 3))
print()
print("When to log-transform in health data:")
print("  - Cost/charge data (always right-skewed)")
print("  - LOS when tail is very long (e.g. >30-day stays)")
print("  - Lab values that span orders of magnitude (creatinine, troponin)")
print("  - Report back-transformed geometric mean for interpretation")


## 5. Age-Standardisation (Direct Method)

In [ ]:
# Direct age-standardisation: applies study rates to a standard population
# to remove confounding by age structure differences between groups.

# US 2000 Standard Population weights by age group
std_pop = pd.DataFrame({
    'age_group' : ['18-44','45-64','65-74','75+'],
    'std_weight': [0.3761, 0.3399, 0.1267, 0.0573]   # proportions sum to ~1
})

# Crude rates by age group × payer (Medicare vs Private as example)
for payer in ['Medicare','Private']:
    sub = df[df['payer'] == payer]
    rates = (sub.groupby('age_group')['readmitted_30']
               .agg(['mean','count'])
               .rename(columns={'mean':'crude_rate','count':'n'})
               .reset_index())
    merged = rates.merge(std_pop, on='age_group')
    merged['weighted_rate'] = merged['crude_rate'] * merged['std_weight']
    age_adj_rate = merged['weighted_rate'].sum() * 100
    crude_overall = sub['readmitted_30'].mean() * 100
    print(f"{payer:10s} — Crude: {crude_overall:.1f}%  |  Age-adjusted: {age_adj_rate:.1f}%")


## 6. Table 1 — Publication-Ready Descriptive Table

A "Table 1" in clinical research summarises the baseline characteristics of a cohort, often stratified by a key variable (e.g. readmitted vs not).


In [ ]:
def table1(df, stratify_col, continuous_cols, categorical_cols):
    """
    Build a basic Table 1.
    Continuous: mean ± SD (or median [IQR] if skewed)
    Categorical: N (%)
    """
    groups = df[stratify_col].unique()
    rows = []

    # Header row
    header = {'Variable': 'Variable'}
    for g in sorted(groups):
        n = (df[stratify_col]==g).sum()
        header[str(g)] = f'{"Not readmitted" if g==0 else "Readmitted"} (N={n})'
    rows.append(header)

    # Continuous variables
    for col in continuous_cols:
        sk = abs(skew(df[col].dropna()))
        row = {'Variable': col + (' [median, IQR]' if sk > 1 else ' [mean ± SD]')}
        for g in sorted(groups):
            vals = df.loc[df[stratify_col]==g, col].dropna()
            if sk > 1:
                row[str(g)] = f"{vals.median():.1f} [{vals.quantile(.25):.1f}–{vals.quantile(.75):.1f}]"
            else:
                row[str(g)] = f"{vals.mean():.1f} ± {vals.std():.1f}"
        rows.append(row)

    # Categorical variables
    for col in categorical_cols:
        row = {'Variable': col}
        for g in sorted(groups):
            sub = df[df[stratify_col]==g]
            row[str(g)] = ''
        rows.append(row)
        for cat in df[col].dropna().unique():
            row = {'Variable': f'  {cat}'}
            for g in sorted(groups):
                sub = df[df[stratify_col]==g]
                n_cat = (sub[col]==cat).sum()
                pct   = n_cat / len(sub) * 100
                row[str(g)] = f'{n_cat} ({pct:.1f}%)'
            rows.append(row)

    return pd.DataFrame(rows).set_index('Variable')

t1 = table1(
    df,
    stratify_col    = 'readmitted_30',
    continuous_cols = ['age','los_days','total_charge'],
    categorical_cols= ['sex','payer','admit_type','age_group']
)
print("Table 1 — Baseline characteristics by 30-day readmission status:")
display(t1)


## 7. Statistical Tests for Group Comparisons

In [ ]:
from scipy.stats import chi2_contingency, mannwhitneyu, ttest_ind

print("Statistical tests: readmitted vs not readmitted\n")

# --- Continuous: LOS (skewed → Mann-Whitney U) ---
g0 = df.loc[df['readmitted_30']==0, 'los_days']
g1 = df.loc[df['readmitted_30']==1, 'los_days']
stat, p = mannwhitneyu(g0, g1, alternative='two-sided')
print(f"LOS — Mann-Whitney U: U={stat:.0f}, p={p:.4f}")

# --- Continuous: Age (normal → t-test) ---
g0a = df.loc[df['readmitted_30']==0, 'age']
g1a = df.loc[df['readmitted_30']==1, 'age']
t, p_t = ttest_ind(g0a, g1a)
print(f"Age — Student t-test: t={t:.3f}, p={p_t:.4f}")

# --- Categorical: Sex × Readmission (chi-square) ---
ct = pd.crosstab(df['sex'], df['readmitted_30'])
chi2, p_chi, dof, expected = chi2_contingency(ct)
print(f"Sex — Chi-square: χ²={chi2:.3f}, df={dof}, p={p_chi:.4f}")

# --- Categorical: Payer × Readmission ---
ct2 = pd.crosstab(df['payer'], df['readmitted_30'])
chi2_p, p_p, _, _ = chi2_contingency(ct2)
print(f"Payer — Chi-square: χ²={chi2_p:.3f}, p={p_p:.4f}")

print("\n(p < 0.05 suggests significant difference between groups)")


## 8. Knowledge Check

1. Why is the median preferred over the mean for LOS and charge data?
2. What does a readmission rate of 14 per 100 discharges mean clinically?
3. When would you use age-standardisation instead of crude rates?
4. In a chi-square test, what does a p-value of 0.03 for payer × readmission tell you?
5. Re-calculate Table 1 stratified by `sex` instead of `readmitted_30`.


In [ ]:
# --- Exercise: Table 1 stratified by sex ---
t1_sex = table1(
    df,
    stratify_col    = 'sex',
    continuous_cols = ['age','los_days','total_charge'],
    categorical_cols= ['payer','admit_type','age_group','readmitted_30']
)
display(t1_sex)


---
## Summary

In this notebook you:
- Calculated crude prevalence, readmission rates, and Wilson 95% CIs
- Built stratified summaries across payer, age group, and sex
- Diagnosed and handled right-skewed distributions (log-transform)
- Applied direct age-standardisation to remove age confounding
- Constructed a publication-style Table 1
- Ran appropriate statistical tests for continuous and categorical variables

**Next:** `NB-03 · Visualising Clinical Distributions` — histograms, box plots, violin plots, heatmaps, and annotated reference-range plots.
